# Business Questions 

In [0]:
%sql
-- Customer Purchasing Behavior Analysis by Day of Week and Hour of Day
-- Using the Instacart Gold Layer tables

WITH order_counts AS (
  SELECT 
    t.order_dow,
    t.day_name,
    t.order_hour_of_day,
    t.is_weekend,
    COUNT(DISTINCT f.order_id) as total_orders,
    COUNT(f.product_id) as total_items,
    AVG(f.add_to_cart_order) as avg_cart_position,
    SUM(CASE WHEN f.reordered THEN 1 ELSE 0 END) as reordered_items,
    COUNT(f.product_id) as total_items_for_pct
  FROM instacart.instacart_gold.fact_order_items f
  INNER JOIN instacart.instacart_gold.dim_order_time t 
    ON f.timekey = t.order_time_key
  WHERE t.is_current = TRUE
  GROUP BY 
    t.order_dow,
    t.day_name,
    t.order_hour_of_day,
    t.is_weekend
)
SELECT 
  order_dow,
  day_name,
  order_hour_of_day,
  is_weekend,
  total_orders,
  total_items,
  ROUND(total_items * 1.0 / total_orders, 2) as avg_items_per_order,
  ROUND(reordered_items * 100.0 / total_items_for_pct, 2) as reorder_rate_pct,
  ROUND(avg_cart_position, 2) as avg_cart_position
FROM order_counts
ORDER BY order_dow, order_hour_of_day

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Get the data from the previous SQL query and convert to pandas
df = _sqldf.toPandas()

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

# Create a 2x2 subplot layout
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Heatmap: Orders by Day of Week and Hour
orders_pivot = df.pivot_table(
    values='total_orders', 
    index='day_name', 
    columns='order_hour_of_day', 
    aggfunc='sum'
)
# Reorder days of week
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
orders_pivot = orders_pivot.reindex([d for d in day_order if d in orders_pivot.index])

sns.heatmap(orders_pivot, annot=False, fmt='g', cmap='YlOrRd', ax=axes[0, 0], cbar_kws={'label': 'Total Orders'})
axes[0, 0].set_title('Order Volume Heatmap: Day of Week vs Hour of Day', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Hour of Day', fontsize=12)
axes[0, 0].set_ylabel('Day of Week', fontsize=12)

# 2. Line chart: Orders by Hour (comparing weekday vs weekend)
weekday_data = df[df['is_weekend'] == False].groupby('order_hour_of_day')['total_orders'].sum()
weekend_data = df[df['is_weekend'] == True].groupby('order_hour_of_day')['total_orders'].sum()

axes[0, 1].plot(weekday_data.index, weekday_data.values, marker='o', linewidth=2, label='Weekday', color='#2E86AB')
axes[0, 1].plot(weekend_data.index, weekend_data.values, marker='s', linewidth=2, label='Weekend', color='#A23B72')
axes[0, 1].set_title('Order Volume by Hour: Weekday vs Weekend', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Hour of Day', fontsize=12)
axes[0, 1].set_ylabel('Total Orders', fontsize=12)
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xticks(range(0, 24, 2))

# 3. Bar chart: Average Items per Order by Day of Week
avg_items_by_day = df.groupby('day_name').agg({
    'total_items': 'sum',
    'total_orders': 'sum'
}).reset_index()
avg_items_by_day['avg_items_per_order'] = avg_items_by_day['total_items'] / avg_items_by_day['total_orders']
avg_items_by_day['day_name'] = pd.Categorical(avg_items_by_day['day_name'], categories=day_order, ordered=True)
avg_items_by_day = avg_items_by_day.sort_values('day_name')

colors = ['#F18F01' if day in ['Saturday', 'Sunday'] else '#2E86AB' for day in avg_items_by_day['day_name']]
axes[1, 0].bar(avg_items_by_day['day_name'], avg_items_by_day['avg_items_per_order'], color=colors, alpha=0.8)
axes[1, 0].set_title('Average Items per Order by Day of Week', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Day of Week', fontsize=12)
axes[1, 0].set_ylabel('Avg Items per Order', fontsize=12)
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Line chart: Reorder Rate by Hour
reorder_by_hour = df.groupby('order_hour_of_day')['reorder_rate_pct'].mean().reset_index()
reorder_by_hour.columns = ['order_hour_of_day', 'reorder_rate']

axes[1, 1].plot(reorder_by_hour['order_hour_of_day'], reorder_by_hour['reorder_rate'], 
                marker='o', linewidth=2, color='#06A77D', markersize=6)
axes[1, 1].set_title('Reorder Rate by Hour of Day', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Hour of Day', fontsize=12)
axes[1, 1].set_ylabel('Reorder Rate (%)', fontsize=12)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("CUSTOMER PURCHASING BEHAVIOR SUMMARY")
print("="*60)

total_orders = df['total_orders'].sum()
total_items = df['total_items'].sum()

print(f"\nTotal Orders: {total_orders:,}")
print(f"Total Items: {total_items:,}")
print(f"Overall Avg Items per Order: {total_items/total_orders:.2f}")

# Peak hour
peak_hour_data = df.groupby('order_hour_of_day')['total_orders'].sum().idxmax()
print(f"\nPeak Hour: {peak_hour_data}:00 - {peak_hour_data+1}:00")

# Peak day
peak_day = df.groupby('day_name')['total_orders'].sum().idxmax()
print(f"Peak Day: {peak_day}")

# Weekend vs Weekday comparison
weekend_orders = df[df['is_weekend'] == True]['total_orders'].sum()
weekday_orders = df[df['is_weekend'] == False]['total_orders'].sum()
print(f"\nWeekday Orders: {weekday_orders:,} ({weekday_orders/total_orders*100:.1f}%)")
print(f"Weekend Orders: {weekend_orders:,} ({weekend_orders/total_orders*100:.1f}%)")

print("\n" + "="*60)

###Business Question
Which products and departments are purchased most frequently?
Objective:
- Join FactOrders with DimProducts.
- Calculate product purchase frequency.
- Calculate department purchase frequency.
- Identify Top 10 products.
- Identify Top 10 departments.
- Support visualization and business insights.


Validation
Expected:
- No NULL product_id values
- No NULL department_id values
- Purchase counts > 0

In [0]:
%sql
-- Product Purchase Frequency
CREATE OR REPLACE VIEW instacart.instacart_gold.vw_product_frequency AS
SELECT
    p.product_id,
    p.product_name,
    COUNT(*) AS purchase_count
FROM instacart.instacart_gold.fact_order_items f
INNER JOIN instacart.instacart_gold.dim_products p
    ON f.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name;
-- Department Purchase Frequency
CREATE OR REPLACE VIEW instacart.instacart_gold.vw_department_frequency AS
SELECT
    p.department_id,
    p.department,
    COUNT(*) AS purchase_count
FROM instacart.instacart_gold.fact_order_items f
INNER JOIN instacart.instacart_gold.dim_products p
    ON f.product_id = p.product_id
GROUP BY
    p.department_id,
    p.department;
-- Top 10 Products
SELECT
    product_id,
    product_name,
    purchase_count
FROM instacart.instacart_gold.vw_product_frequency
ORDER BY purchase_count DESC
LIMIT 10;
-- Top 10 Departments
SELECT
    department_id,
    department,
    purchase_count
FROM instacart.instacart_gold.vw_department_frequency
ORDER BY purchase_count DESC
LIMIT 10;

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
# Load Top 10 Products
df_products = spark.sql("""
SELECT
product_name,
purchase_count
FROM instacart.instacart_gold.vw_product_frequency
ORDER BY purchase_count DESC
LIMIT 10
""").toPandas()
# Load Top 10 Departments
df_departments = spark.sql("""
SELECT
department,
purchase_count
FROM instacart.instacart_gold.vw_department_frequency
ORDER BY purchase_count DESC
LIMIT 10
""").toPandas()
# -----------------------------
# Top 10 Products Visualization
# -----------------------------
plt.figure(figsize=(10, 6))
plt.barh(
df_products["product_name"][::-1],
df_products["purchase_count"][::-1]
)
plt.title("Top 10 Most Frequently Purchased Products")
plt.xlabel("Purchase Count")
plt.ylabel("Product Name")
plt.tight_layout()
plt.show()
# --------------------------------
# Top 10 Departments Visualization
# --------------------------------
plt.figure(figsize=(10, 6))
plt.barh(
df_departments["department"][::-1],
df_departments["purchase_count"][::-1]
)
plt.title("Top 10 Most Frequently Purchased Departments")
plt.xlabel("Purchase Count")
plt.ylabel("Department")
plt.tight_layout()
plt.show()



### Which products have the highest reorder behavior? - Joy


Y-axis: Product name
X-axis: Reorder rate
Color or label: Total reorders
Title:Top 10 Products by Reorder Rate
--------------------------
We identified the products with the highest reorder behavior by calculating the percentage of purchases marked as reordered. To make the ranking more reliable, we only included products purchased at least 20 times. Products with high reorder rates indicate strong repeat demand and customer preference.

In [0]:
%sql
WITH product_reorder_stats AS (
    SELECT
        p.product_id,
        p.product_name,
        COUNT(*) AS total_purchases,
        SUM(CASE WHEN f.reordered THEN 1 ELSE 0 END) AS total_reorders,
        ROUND(
            100.0 * SUM(CASE WHEN f.reordered THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS reorder_rate
    FROM instacart.instacart_gold.fact_order_items AS f
    INNER JOIN instacart.instacart_gold.dim_products AS p
        ON f.product_id = p.product_id
    GROUP BY
        p.product_id,
        p.product_name
)
SELECT
    product_id,
    product_name,
    total_purchases,
    total_reorders,
    reorder_rate
FROM product_reorder_stats
WHERE total_purchases >= 20
ORDER BY
    reorder_rate DESC,
    total_reorders DESC
LIMIT 10;

In [0]:
import matplotlib.pyplot as plt
# Business Question:
# Which products have the highest reorder behavior?
df = spark.sql("""
    SELECT
        p.product_name,
        ROUND(
            AVG(CAST(f.reordered AS DOUBLE)) * 100,
            2
        ) AS reorder_rate
    FROM instacart.instacart_gold.fact_order_items AS f
    JOIN instacart.instacart_gold.dim_products AS p
        ON f.product_id = p.product_id
    GROUP BY p.product_name
    ORDER BY reorder_rate DESC
    LIMIT 10
""").toPandas()
df = df.sort_values("reorder_rate")
plt.figure(figsize=(10, 6))
plt.barh(df["product_name"], df["reorder_rate"], color="skyblue")
plt.title("Which Products Have the Highest Reorder Behavior?")
plt.xlabel("Reorder Rate (%)")
plt.ylabel("Product Name")
plt.tight_layout()
plt.show()

### Which aisles have the highest reorder rate by day of the week? 

 Relevant columns:
 aisle_id → identifies the aisle.
 reordered → flag for reorder behavior.
 timekey → links to Dim_Order_Time, which has order_dow (day of week).

In [0]:
%sql
SELECT
    p.aisle,
    t.order_dow,
    ROUND(AVG(CAST(f.reordered AS DOUBLE)) * 100, 2) AS reorder_rate
FROM instacart.instacart_gold.fact_order_items AS f
JOIN instacart.instacart_gold.dim_order_time AS t
    ON f.timekey = t.order_time_key
JOIN instacart.instacart_gold.dim_products AS p
    ON f.product_id = p.product_id
GROUP BY p.aisle, t.order_dow
ORDER BY reorder_rate DESC
LIMIT 100;

In [0]:
import matplotlib.pyplot as plt
# Business Question:
# Which aisles have the highest reorder rate by day of the week?
df = spark.sql("""
    SELECT
        p.aisle AS aisle_name,
        t.order_dow,
        ROUND(AVG(CASE WHEN f.reordered = TRUE THEN 1 ELSE 0 END) * 100, 2) AS reorder_rate
    FROM instacart.instacart_gold.fact_order_items AS f
    JOIN instacart.instacart_gold.dim_order_time AS t
        ON f.timekey = t.order_time_key
    JOIN instacart.instacart_gold.dim_products AS p
        ON f.product_id = p.product_id
    GROUP BY p.aisle, t.order_dow
    ORDER BY reorder_rate DESC
    LIMIT 100
""").toPandas()
# Sort values so bars look neat
df = df.sort_values("reorder_rate")
# Horizontal bar chart
plt.barh(df["aisle_name"], df["reorder_rate"], color="skyblue")
plt.title(
    "Which Aisles Have the Highest Reorder Rate?\n"
    "Top 10 Aisles by Day of Week"
)
plt.xlabel("Reorder Rate (%)")
plt.ylabel("Aisle Name")
plt.tight_layout()
plt.show()